In [1]:
# 1. Import libraries and helper functions

import pandas as pd
from pathlib import Path
import os

def normalize_text(series):
    return series.astype(str).str.strip().str.title()

In [2]:
# 2. Path configuration

BASE_DIR = Path(os.environ.get("MASKING_DATA_PATH", "../data"))
RAW_DIR = BASE_DIR / "raw" / "us"
PROCESSED_DIR = BASE_DIR / "processed" / "us"

In [3]:
# 3. Load dataset

df = pd.read_csv(RAW_DIR / "reps_raw.csv").dropna(how='all')

In [4]:
# 4. Rename columns  

df = df.rename(columns={
    'REP_ID': 'rep_id',
    'SALES_REP': 'rep_name',
    'MANAGER': 'rep_manager'
})

In [5]:
# 5. Validate before changing anything

assert df['rep_id'].duplicated().sum() == 0, "rep_id duplicated!"
assert -1 not in df['rep_id'].values, "rep_id -1 already exists!"

In [6]:
# 6. Add category 'Unknown' as id -1

unknown_row = pd.DataFrame({
    'rep_id': [-1],
    'rep_name': ['Unknown'],
    'rep_manager': ['Unknown']
})

df = pd.concat([df, unknown_row], ignore_index=True)

In [7]:
# 7. Convert data types

df['rep_id'] = df['rep_id'].astype(str)
df['rep_name'] = normalize_text(df['rep_name'])
df['rep_manager'] = normalize_text(df['rep_manager']).astype('category')
df['rep_manager'] = normalize_text(df['rep_manager']).replace('nan', 'Unknown').astype('category')

In [8]:
# 8. Final checks

assert df['rep_id'].duplicated().sum() == 0, "rep_id duplicated after concat!"
assert df.isna().sum().sum() == 0, "Null values detected!"

In [9]:
# 9. Export cleaned dataset for BI 

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "dim_reps.csv", index=False)

print(f"Success! {len(df)} reps exported, including 1 'Unknown' fallback member.")

Success! 26 reps exported, including 1 'Unknown' fallback member.
